# 序列逆置
使用sequence to sequence 模型将一个字符串序列逆置。
例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个sequence to sequence 模型示意图 )
![seq2seq](./seq2seq.png)

In [9]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
#多import了一个 from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [10]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase #string.ascii_uppercase 是 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
    return ''.join(random.choice(letters) for i in range(stringLength))#，每次随机从中选取一个字母，拼接成字符串。

def get_batch(batch_size, length):
    #生成原始样本
    batched_examples = [randomString(length) for i in range(batch_size)]#batch_size 个随机字符串，每个长度为 length。这些字符串将作为输入序列的原始文本。
    #编码输入--每个字符映射为整数索引：'A' → 1，'B' → 2，……，'Z' → 26。
    #结果是一个二维列表，形状为 (batch_size, length)，存储编码后的输入序列。
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    #目标序列--每个输入序列反转作为目标序列。
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    #解码器输入--对于每个目标序列，在前面添加一个起始标记（0），并去掉最后一个字符。
    #这是Teacher Forcing的典型做法：解码器的每一步输入是目标序列的前一个词（或起始标记），用于预测下一个词。
    #假设目标 y = [3,2,1]，则 dec_x = [0,3,2]。
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['RTIEYRLUSE', 'PTMGWAULQZ'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[18, 20,  9,  5, 25, 18, 12, 21, 19,  5],
       [16, 20, 13,  7, 23,  1, 21, 12, 17, 26]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0,  5, 19, 21, 12, 18, 25,  5,  9, 20],
       [ 0, 26, 17, 12, 21,  1, 23,  7, 13, 20]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 5, 19, 21, 12, 18, 25,  5,  9, 20, 18],
       [26, 17, 12, 21,  1, 23,  7, 13, 20, 16]], dtype=int32)>)


# 建立sequence to sequence 模型

In [11]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27 #词汇表大小：26个大写字母 + 1个起始符(0)
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64, 
                                                    input_shape=(None,))#batch_input_shape=[None, None])
        
        #定义两个 SimpleRNNCell，隐藏单元数为 128。它们将用于构建编码器和解码器的循环单元。
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(128)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(128)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        
        #全连接层，将解码器的输出映射到词汇表大小，用于计算每个词的概率分布。
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
    def call(self, enc_ids, dec_ids):
        '''
        完成sequence2sequence 模型的搭建，模块已经在`__init__`函数中定义好
        '''
        #编码
        enc_state = self.encode(enc_ids)   # enc_state: (batch, 128)

        #解码器输入嵌入
        dec_emb = self.embed_layer(dec_ids)   # (batch, dec_len, 64)

        #解码器前向，传入初始状态
        dec_out, _ = self.decoder(dec_emb, initial_state=[enc_state])   # dec_out: (batch, dec_len, 128)

        #映射到词汇表
        logits = self.dense(dec_out)          # (batch, dec_len, v_sz)
        return logits
    
    
#     @tf.function
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)
        _, enc_state = self.encoder(enc_emb)
        return enc_state
    
    def get_next_token(self, x, state):
        '''
        shape(x) = [b_sz,] 
        '''
        inp_emb = self.embed_layer(x) #shape(b_sz, emb_sz)
        h, [state] = self.decoder_cell(inp_emb, [state]) # shape(b_sz, h_sz)
        logits = self.dense(h) # shape(b_sz, v_sz)
        out = tf.argmax(logits, axis=-1)
        return out, state

# Loss函数以及训练逻辑

In [12]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    grads_and_vars = [(g, v) for g, v in zip(grads, model.trainable_variables) if g is not None]
    if not grads_and_vars:
        raise ValueError('No gradients found. Please check model forward path and loss.')
    optimizer.apply_gradients(grads_and_vars)
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(3000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [14]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
#先做一次前向，确保变量已创建，再进入训练--这套作业代码全是这里有问题！
_, warmup_enc_x, warmup_dec_x, _ = get_batch(2, 20)
_ = model(warmup_enc_x, warmup_dec_x)
train(model, optimizer, seqlen=20)

step 0 : loss 3.3053863
step 500 : loss 1.5113138
step 1000 : loss 0.94440174
step 1500 : loss 0.64534086
step 2000 : loss 0.49135065
step 2500 : loss 0.41188994


<tf.Tensor: shape=(), dtype=float32, numpy=0.34505683183670044>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [15]:
def sequence_reversal():
    def decode(init_state, steps=10):
        b_sz = tf.shape(init_state)[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 10)
    state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1]), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]
[('FQWSOISFHG', 'GHFSIOSWQF'), ('XXZSFQSUTY', 'YTUSQFSZXX'), ('SMDGTTVMMW', 'WMMVTTGDMS'), ('XVMTXCTSAB', 'BASTCXTMVX'), ('RJHYIKLMOK', 'KOMLKIYHJR'), ('VZOHHYGDWF', 'FWDGYHHOZV'), ('LDINCSZJVM', 'MVJZSCNIDL'), ('GUTFCAZVDM', 'MDVZACFTUG'), ('UHSZTMSMCX', 'XCMSMTZSHU'), ('QYZFDWSQKP', 'PKQSWDFZYQ'), ('SESLAKDSJD', 'DJSDKALSES'), ('TWYELERTPG', 'GPTRELEYWT'), ('FBBTXGDJYI', 'IYJDGXTBBF'), ('HHNFLLHUEY', 'YEUHLLFNHH'), ('BFMINZVAJA', 'AJAVZNIMFB'), ('ZFRFDQRGQL', 'LQGRQDFRFZ'), ('MEQLXNBGNZ', 'ZNGBNXLQEM'), ('RPBGVQWCRT', 'TRCWQVGBPR'), ('KFVWSOIWCA', 'ACWIOSWVFK'), ('TTGWLEFEHR', 'RHEFELWGTT'), ('NXXPHULYKF', 'FKYLUHPXXN'), ('LUHBIBAEHX', 'XHEABIBHUL'), ('TYXJMBVCMG', 'GMCVBMJXYT'), ('EHPADMXHNU', 'UNHXMDAPHE'), ('LPCKVYGEDM', 'MDEGYVKCPL'), ('MIAQMFIAHD', 'DHAIFMQAIM'), ('YDOINIDJHP', 'PHJDINIODY